In [ ]:
import networkx as nx
from sage.graphs.graph_coloring import first_coloring, all_graph_colorings, chromatic_number
from sage.graphs.planarity import is_planar
from sage.all import Graph
import json
import numpy as np

In [ ]:
# Read networkx graph
G_nx = nx.read_graphml("graph.graphml", node_type=int)

# Convert networkx graph to sagegraph
G_sage = Graph()
G_sage.add_vertices(G_nx.nodes())
G_sage.add_edges(
    (u, v, data.copy())
    for u, v, data in G_nx.edges(data=True)
)

In [ ]:
# create array of areas for equitable coloring
# sage graph does not store node attributes
areas = np.zeros(G_nx.order())
for node, data in G_nx.nodes(data=True):
    areas[node] = data["area"]

In [ ]:
# Check graph stats
def graph_stats(G):
    print(f"Is planar: {is_planar(G)}")
    print(f"number of nodes: {G.order()}")
    print(f"number of edges: {G.size()}")

In [ ]:
graph_stats(G_sage)

In [ ]:
# Remove edges that are below a certain length to reduce complexity of the graph
# This step is required otherwise the algorithm takes too long

max_length = 10.8

H_sage = G_sage.copy()
H_sage.delete_edges([
    (u, v)
    for u, v, data in H_sage.edges()
    if data['length'] < max_length
])

graph_stats(H_sage)

## First coloring

In [ ]:
%%time
x = first_coloring(H_sage, 4)

## Equitable coloring

In [ ]:
all_colorings = all_graph_colorings(H_sage, 4)

In [ ]:
i = 0
colorings = []
min_count_diff = 999999999999
min_area_diff = 999999999999
for c in all_colorings:
    
    i += 1
    if i > 10000:
        break
    total_counts = [len(c[color]) for color in c]
    total_areas = [areas[c[color]].sum() for color in c]
    count_diff = max(total_counts) - min(total_counts)
    area_diff = max(total_areas) - min(total_areas)
    
    if area_diff < min_area_diff:
        min_area_diff = area_diff
        print(f"area diff: {area_diff}. count diff: {count_diff}")
        colorings.append(c)

In [ ]:
colorings[-1]

In [ ]:
G_colored = G_nx.copy()
coloring = colorings[-1]
for color, nodes in coloring.items():
    nx.set_node_attributes(G_colored, {node: color for node in nodes}, "color")
nx.write_graphml(G_colored, 'graph_colored.graphml')